
 ## SparqlCE training script — DBpedia.

Reproduces the paper's main-branch recipe:
    loss      = InfoNCE(in-batch) + 0.3 * MSE(cos(a,p), jaccard(a,p))
    optimizer = AdamW, differential LRs (CodeBERT 1e-5, GNN/proj 1e-4)
                (or single LR 1e-5 — flip USE_DIFFERENTIAL_LR below)
    selection = best val MRR  (NOT mean cosine — that's biased toward collapse)
    splits    = paper's fixed split if available, else seeded random split

Run:
    python train_dbpedia.py

Before running, VERIFY the three paths in the CONFIG block.
"""



In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "sparqlce").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)
print("sparqlce exists:", (ROOT / "sparqlce").exists())

Project root: /local/upb/users/z/zahera/profiles/unix/cs/SPARQL-Embeddings
sparqlce exists: True


In [2]:
import importlib
import json
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, random_split
from tqdm.auto import tqdm

import sparqlce.graph_builder
import sparqlce.encoder
import sparqlce.dataset

importlib.reload(sparqlce.graph_builder)
importlib.reload(sparqlce.encoder)
importlib.reload(sparqlce.dataset)

from sparqlce.config import Config
from sparqlce.graph_builder import SPARQLGraphBuilder
from sparqlce.encoder import HybridSPARQLEncoder
from sparqlce.dataset import SemanticSPARQLDataset, semantic_collate_fn_inbatch

/upb/users/z/zahera/profiles/unix/cs/.conda/envs/sparql-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/upb/users/z/zahera/profiles/unix/cs/.conda/envs/sparql-env/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: /local/upb/users/z/zahera/profiles/unix/cs/.conda/envs/sparql-env/lib/python3.10/site-packages/torch_scatter/_version_cuda.so
  import torch_geometric.typing
/upb/users/z/zahera/profiles/unix/cs/.conda/envs/sparql-env/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: Could not load this library: /local/upb/users/z/zahera/profiles/unix/cs/.conda/envs/s

In [3]:
# ============================================================================
# 1. Config
# ============================================================================

SEED = 42

DATA_PATH = "../../datasets/dbpedia-triplets-dataset.json"
SPLITS_PATH = "../../datasets/splits_dbpedia.json"
CKPT_DIR = "../../checkpoints/sparqlce_dbpedia"

BATCH_SIZE = 32
EPOCHS = 20
TEMPERATURE = 0.05
LAMBDA_REG = 0.3
GRAD_CLIP = 1.0
WEIGHT_DECAY = 1e-2

USE_DIFFERENTIAL_LR = True
LR_TEXT_ENCODER = 1e-5
LR_REST = 1e-4
LR_SINGLE = 1e-5

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("device:", device)
print("DATA_PATH:", DATA_PATH)
print("SPLITS_PATH:", SPLITS_PATH)
print("CKPT_DIR:", CKPT_DIR)

device: cuda
DATA_PATH: ../../datasets/dbpedia-triplets-dataset.json
SPLITS_PATH: ../../datasets/splits_dbpedia.json
CKPT_DIR: ../../checkpoints/sparqlce_dbpedia


In [4]:
# ============================================================================
# 2. Data
# ============================================================================

graph_builder = SPARQLGraphBuilder()

dataset = SemanticSPARQLDataset(
    DATA_PATH,
    graph_builder=graph_builder,
    in_batch_negatives=True,
)

print(f"Loading dataset from {DATA_PATH}...")
print(f"Loaded {len(dataset)} anchor groups.")

if os.path.exists(SPLITS_PATH):
    with open(SPLITS_PATH, "r") as f:
        splits = json.load(f)

    train_idx = splits.get("train_idx", splits.get("train"))
    val_idx = splits.get("val_idx", splits.get("valid_idx", splits.get("val")))
    test_idx = splits.get("test_idx", splits.get("test"))

    assert train_idx is not None and val_idx is not None and test_idx is not None, \
        f"Could not parse splits from {SPLITS_PATH}"

    train_dataset = Subset(dataset, train_idx)
    val_dataset = Subset(dataset, val_idx)
    test_dataset = Subset(dataset, test_idx)

    print(f"✓ Using FIXED split from {SPLITS_PATH}")
else:
    print(f"⚠ Fixed split file not found at {SPLITS_PATH}")
    print("⚠ Falling back to seeded random split.")
    total = len(dataset)
    n_train = int(0.80 * total)
    n_val = int(0.10 * total)
    n_test = total - n_train - n_val
    gen = torch.Generator().manual_seed(SEED)

    train_dataset, val_dataset, test_dataset = random_split(
        dataset, [n_train, n_val, n_test], generator=gen
    )

print(f" train={len(train_dataset)} val={len(val_dataset)} test={len(test_dataset)}")

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=semantic_collate_fn_inbatch,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=semantic_collate_fn_inbatch,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=semantic_collate_fn_inbatch,
)

Loading dataset from ../../datasets/dbpedia-triplets-dataset.json...
Loaded 4713 anchor groups.
Loading dataset from ../../datasets/dbpedia-triplets-dataset.json...
Loaded 4713 anchor groups.
✓ Using FIXED split from ../../datasets/splits_dbpedia.json
 train=3770 val=471 test=472


In [5]:
# ============================================================================
# 3. Graph builder sanity check
# ============================================================================

q = 'SELECT * WHERE { ?x wdt:P31/wdt:P279* wd:Q5 . OPTIONAL { ?x rdfs:label ?l } FILTER(?x != ?l) }'
G = graph_builder.build_graph(q)

print("nodes:", G.number_of_nodes(), "edges:", G.number_of_edges())
print("sample nodes:", list(G.nodes(data=True))[:8])
print("sample edges:", list(G.edges(data=True))[:8])

nodes: 6 edges: 9
sample nodes: [(0, {'label': 'VAR:?v0'}), (1, {'label': 'ENT:wd:Q5'}), (2, {'label': 'PATH:wdt:P279*'}), (3, {'label': 'VAR:?v1'}), (4, {'label': 'FILTER:?v0 != ?v1'}), (5, {'label': 'OPTIONAL'})]
sample edges: [(0, 2, {'label': 'path_in'}), (0, 3, {'label': 'PRED:rdfs:label'}), (0, 4, {'label': 'uses_filter'}), (1, 3, {'label': 'join'}), (2, 1, {'label': 'path_out'}), (3, 1, {'label': 'join'}), (3, 4, {'label': 'uses_filter'}), (5, 0, {'label': 'optional_scope'})]


In [6]:
# ============================================================================
# 4. Model + optimizer
# ============================================================================

model = HybridSPARQLEncoder(ablation="full").to(device)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"total params:     {total:,}")
print(f"trainable params: {trainable:,}")
print(f"trainable %:      {100 * trainable / total:.2f}")

if USE_DIFFERENTIAL_LR:
    text_kw = ("text", "txt", "codebert", "bert", "text_encoder", "backbone")
    text_params = [
        p for n, p in model.named_parameters()
        if any(k in n.lower() for k in text_kw) and p.requires_grad
    ]
    rest_params = [
        p for n, p in model.named_parameters()
        if not any(k in n.lower() for k in text_kw) and p.requires_grad
    ]

    n_text = sum(p.numel() for p in text_params)
    n_rest = sum(p.numel() for p in rest_params)

    print(f"✓ Differential LRs: text={n_text:,} params @ {LR_TEXT_ENCODER}, other={n_rest:,} params @ {LR_REST}")

    if n_text == 0:
        print("⚠ No params matched text-encoder keywords; using single LR fallback.")
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=LR_SINGLE,
            weight_decay=WEIGHT_DECAY,
        )
    else:
        optimizer = torch.optim.AdamW(
            [
                {"params": text_params, "lr": LR_TEXT_ENCODER},
                {"params": rest_params, "lr": LR_REST},
            ],
            weight_decay=WEIGHT_DECAY,
        )
else:
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR_SINGLE,
        weight_decay=WEIGHT_DECAY,
    )

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1376.12it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


total params:     115,983,488
trainable params: 115,983,488
trainable %:      100.00
✓ Differential LRs: text=109,580,544 params @ 1e-05, other=6,402,944 params @ 0.0001


In [7]:
# ============================================================================
# helper: unwrap model outputs
# ============================================================================

def extract_embedding(model_output):
    if isinstance(model_output, torch.Tensor):
        return model_output

    if isinstance(model_output, (list, tuple)):
        for x in reversed(model_output):
            if isinstance(x, torch.Tensor) and x.dim() == 2:
                return x
        for x in model_output:
            if isinstance(x, torch.Tensor):
                return x

    if isinstance(model_output, dict):
        for key in ["embedding", "embeddings", "query_embedding", "fused_embedding", "fused"]:
            if key in model_output and isinstance(model_output[key], torch.Tensor):
                return model_output[key]
        for v in model_output.values():
            if isinstance(v, torch.Tensor) and v.dim() == 2:
                return v

    raise TypeError(f"Could not extract embedding tensor from model output of type {type(model_output)}")

In [8]:
# ============================================================================
# 5. Losses
# ============================================================================

def infonce_jaccard_loss(a_emb, p_emb, pos_js, temperature=TEMPERATURE, lambda_reg=LAMBDA_REG):
    a = F.normalize(a_emb, dim=1)
    p = F.normalize(p_emb, dim=1)

    logits = a @ p.T / temperature
    logits = logits + torch.diag(pos_js.clamp(0.0, 1.0))
    labels = torch.arange(a.size(0), device=a.device)

    loss_ce = F.cross_entropy(logits, labels)

    cos_raw = (a * p).sum(dim=1)
    cos_scaled = (cos_raw + 1.0) / 2.0
    loss_reg = F.mse_loss(cos_scaled, pos_js)

    loss = loss_ce + lambda_reg * loss_reg
    return loss, loss_ce, loss_reg

In [9]:
# ============================================================================
# 6. Validation
# ============================================================================

@torch.no_grad()
def evaluate_ranking(model, loader, device):
    model.eval()
    anchors, positives = [], []

    for batch in loader:
        a_emb, _ = model(batch[0].to(device))
        p_emb, _ = model(batch[1].to(device))
        anchors.append(a_emb.cpu())
        positives.append(p_emb.cpu())

    A = torch.cat(anchors)
    P = torch.cat(positives)

    sims = A @ P.T
    diag = sims.diag().unsqueeze(1)
    ranks = (sims >= diag).sum(dim=1).float()

    return {
        "mrr": (1.0 / ranks).mean().item(),
        "hits@1": (ranks == 1).float().mean().item(),
        "hits@5": (ranks <= 5).float().mean().item(),
    }

In [10]:
# ============================================================================
# 7. Training loop
# ============================================================================

os.makedirs(CKPT_DIR, exist_ok=True)
best_mrr = -1.0

print(f"\n✓ Training on {device} for {EPOCHS} epochs")
print(f" loss = InfoNCE(τ={TEMPERATURE}) + {LAMBDA_REG} * MSE((cos+1)/2, jaccard)")

for epoch in range(1, EPOCHS + 1):
    model.train()
    sum_loss, sum_ce, sum_reg = 0.0, 0.0, 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{EPOCHS}")

    for batch in pbar:
        a_graphs = batch[0].to(device)
        p_graphs = batch[1].to(device)
        pos_js = batch[2].to(device)

        optimizer.zero_grad()

        a_emb, _ = model(a_graphs)
        p_emb, _ = model(p_graphs)

        loss, loss_ce, loss_reg = infonce_jaccard_loss(
            a_emb, p_emb, pos_js,
            temperature=TEMPERATURE,
            lambda_reg=LAMBDA_REG,
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        optimizer.step()

        sum_loss += loss.item()
        sum_ce += loss_ce.item()
        sum_reg += loss_reg.item()

        pbar.set_postfix(
            loss=f"{loss.item():.4f}",
            ce=f"{loss_ce.item():.4f}",
            reg=f"{loss_reg.item():.4f}",
        )

    nb = max(1, len(train_loader))
    val = evaluate_ranking(model, val_loader, device)

    print(
        f"Epoch {epoch:02d} | train_loss={sum_loss/nb:.4f} "
        f"(ce={sum_ce/nb:.4f} reg={sum_reg/nb:.4f}) "
        f"| val MRR={val['mrr']:.4f} Hits@1={val['hits@1']:.4f} Hits@5={val['hits@5']:.4f}"
    )

    if val["mrr"] > best_mrr:
        best_mrr = val["mrr"]
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "val_metrics": val,
            },
            os.path.join(CKPT_DIR, "best_model.pt"),
        )
        print(f" ✓ best saved (val MRR={best_mrr:.4f})")


✓ Training on cuda for 20 epochs
 loss = InfoNCE(τ=0.05) + 0.3 * MSE((cos+1)/2, jaccard)


Epoch 01/20: 100%|██████████| 118/118 [01:19<00:00,  1.48it/s, ce=0.9333, loss=0.9346, reg=0.0043]


Epoch 01 | train_loss=1.0509 (ce=1.0491 reg=0.0059) | val MRR=0.3948 Hits@1=0.3015 Hits@5=0.4628
 ✓ best saved (val MRR=0.3948)


Epoch 02/20: 100%|██████████| 118/118 [01:19<00:00,  1.48it/s, ce=0.6380, loss=0.6394, reg=0.0047]


Epoch 02 | train_loss=0.7624 (ce=0.7605 reg=0.0062) | val MRR=0.3977 Hits@1=0.2930 Hits@5=0.4926
 ✓ best saved (val MRR=0.3977)


...

Epoch 18 | train_loss=0.1362 (ce=0.1342 reg=0.0066) | val MRR=0.5598 Hits@1=0.4289 Hits@5=0.6985


Epoch 19/20: 100%|██████████| 118/118 [01:19<00:00,  1.48it/s, ce=0.0772, loss=0.0792, reg=0.0068]


Epoch 19 | train_loss=0.1469 (ce=0.1451 reg=0.0062) | val MRR=0.5412 Hits@1=0.4034 Hits@5=0.7070


Epoch 20/20: 100%|██████████| 118/118 [01:19<00:00,  1.48it/s, ce=0.0360, loss=0.0377, reg=0.0056]


Epoch 20 | train_loss=0.1313 (ce=0.1295 reg=0.0063) | val MRR=0.5774 Hits@1=0.4437 Hits@5=0.7325
 ✓ best saved (val MRR=0.5774)


In [11]:

# ============================================================================
# 8. Final evaluation on the test split
# ============================================================================
print(f"\n✓ Training complete. Best val MRR: {best_mrr:.4f}")
print("✓ Reloading best checkpoint and evaluating on test split...")

best = torch.load(os.path.join(CKPT_DIR, "best_model.pt"), map_location=device)
model.load_state_dict(best["model_state_dict"])
test = evaluate_ranking(model, test_loader, device)
print(f"TEST  MRR={test['mrr']:.4f}  Hits@1={test['hits@1']:.4f}  Hits@5={test['hits@5']:.4f}")


✓ Training complete. Best val MRR: 0.5774
✓ Reloading best checkpoint and evaluating on test split...
TEST  MRR=0.5716  Hits@1=0.4407  Hits@5=0.7352


In [13]:
# ============================================================================
# 9. Save the Final Model
# ============================================================================
# Create a 'models' directory in your project root if it doesn't exist
save_dir = CKPT_DIR
os.makedirs(save_dir, exist_ok=True)

# Define the final path (you can add a timestamp or metric to the filename)
final_model_path = os.path.join(save_dir, f"final_model_mrr_{test['mrr']:.4f}.pt")

# Save the weights
torch.save(model.state_dict(), final_model_path)
print(f"✓ Final model successfully saved to: {final_model_path}")

✓ Final model successfully saved to: ../../checkpoints/sparqlce_dbpedia/final_model_mrr_0.5716.pt
